<a href="https://colab.research.google.com/github/AbhirathA/NLPDL/blob/main/NLPDL_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import numpy
import math

Question 1

Weights assumed:

In [18]:
import numpy
Wq = numpy.array([[1,1,0],[0,0,1],[1,0,1]])
Wk = numpy.array([[0,1,1],[0,1,0],[1,1,1]])
Wv = numpy.array([[1,0,0],[0,1,1],[0,1,1]])

norm = math.sqrt(3)

Embeddings Assumed:


In [19]:
Cat = numpy.array([1,1,0])
Chasing = numpy.array([0,1,0])
Dog = numpy.array([0.8,1,0.6])

Attention Mechanism:

In [20]:
sentence = numpy.array([Cat,Chasing,Dog])
print(sentence)

[[1.  1.  0. ]
 [0.  1.  0. ]
 [0.8 1.  0.6]]


In [21]:
queries = sentence @ Wq
print(queries)

[[1.  1.  1. ]
 [0.  0.  1. ]
 [1.4 0.8 1.6]]


In [22]:
keys = sentence @ Wk
print(keys)

[[0.  2.  1. ]
 [0.  1.  0. ]
 [0.6 2.4 1.4]]


In [23]:
values = sentence @ Wv
print(values)

[[1.  1.  1. ]
 [0.  1.  1. ]
 [0.8 1.6 1.6]]


In [24]:
alpha_scores = (queries @ keys.T) / norm
print(alpha_scores)

[[1.73205081 0.57735027 2.54034118]
 [0.57735027 0.         0.80829038]
 [1.84752086 0.46188022 2.88675135]]


In [25]:
attention_weights = numpy.exp(alpha_scores) / numpy.sum(numpy.exp(alpha_scores), axis=1, keepdims=True)
print(attention_weights)

[[0.28096043 0.08854521 0.63049436]
 [0.35446315 0.19898991 0.44654693]
 [0.24526611 0.06135662 0.69337727]]


In [26]:
final_embeddings = attention_weights @ values
print(final_embeddings)

[[0.78535592 1.37829662 1.37829662]
 [0.7117007  1.26792816 1.26792816]
 [0.79996792 1.41602636 1.41602636]]


New Embeddings:

In [27]:
print("Cat:", final_embeddings[0])
print("Chasing:", final_embeddings[1])
print("Dog:", final_embeddings[2])

Cat: [0.78535592 1.37829662 1.37829662]
Chasing: [0.7117007  1.26792816 1.26792816]
Dog: [0.79996792 1.41602636 1.41602636]


Question 2

In [28]:
class SelfAttention:
  def __init__(self, dim, mask = None):
    self.dim = dim
    self.Wq = numpy.random.randn(dim, dim)
    self.Wk = numpy.random.randn(dim, dim)
    self.Wv = numpy.random.randn(dim, dim)
    self.norm = math.sqrt(dim)
    self.mask = mask

  def forward(self, x):
    self.queries = x @ self.Wq
    self.keys = x @ self.Wk
    self.values = x @ self.Wv

    alpha_scores = (self.queries @ self.keys.T) / self.norm

    if self.mask is not None:
      alpha_scores[self.mask == 0] = -numpy.inf

    attention_weights = numpy.exp(alpha_scores)/numpy.sum(numpy.exp(alpha_scores), axis=1, keepdims=True)

    self.output = attention_weights @ self.values
    return self.output

In [29]:
def positional_encoding(sequence_length, dim):

    position = numpy.arange(sequence_length)[:, numpy.newaxis]
    div_term = numpy.exp(numpy.arange(0, dim, 2) * -(numpy.log(10000.0) / dim))

    pe = numpy.zeros((sequence_length, dim))
    pe[:, 0::2] = numpy.sin(position * div_term)
    pe[:, 1::2] = numpy.cos(position * div_term)
    return pe

In [30]:
class MyTransformerModel:
  def __init__(self, dim, pred_dim):
    self.dim = dim
    self.pred_dim = pred_dim
    self.self_attn = SelfAttention(dim)

    self.ffn_w1 = numpy.random.randn(dim, pred_dim)
    self.ffn_b1 = numpy.random.randn(pred_dim)

    self.activation = lambda x: numpy.maximum(0, x)

    self.ffn_w2 = numpy.random.randn(pred_dim, dim)
    self.ffn_b2 = numpy.random.randn(dim)

    self.layernorm = lambda x: (x - numpy.mean(x, axis=-1, keepdims=True)) / (numpy.std(x, axis=-1, keepdims=True) + 1e-6)

  def forward(self, x):
    sequence_length, dim = x.shape

    pe = positional_encoding(sequence_length, dim)
    x_pe = x + pe

    attn_output = self.self_attn.forward(x_pe)

    x_attn_norm = self.layernorm(x_pe + attn_output)

    ffn_output = x_attn_norm @ self.ffn_w1 + self.ffn_b1
    ffn_output = self.activation(ffn_output)
    ffn_output = ffn_output @ self.ffn_w2 + self.ffn_b2

    output = self.layernorm(x_attn_norm + ffn_output)

    return output